# Ulysses Narrative State Analysis & Stream-of-Consciousness Generation

**Pipeline Overview:**
1. **GMM Analysis** — Cluster 555 annotated text spans from Joyce's *Ulysses* into 8 latent narrative states using Gaussian Mixture Models
2. **SFT (LoRA)** — Fine-tune Gemma-2-2b-it via Unsloth/QLoRA with discrete control tokens (`<CTRL|EP=...|STATE=...|ENT=...|>`)
3. **Controlled Generation** — Generate text in specified consciousness states, including anachronistic topics
4. **Dynamic Flow** — Chain multiple states to simulate transitions in stream of consciousness

**Requirements:** Google Colab (T4/A100 GPU), Google Drive for data files.

---

## 0. Environment Setup

In [16]:
# ==========================================
# 0. Environment Setup & Dependency Install
# ==========================================
# Run this cell first. If pyarrow errors occur, restart runtime and re-run.

import os, sys

try:
    import unsloth
    print("Dependencies already installed.")
except ImportError:
    print("Installing dependencies...")
    !pip install "pyarrow>=15.0.0" --force-reinstall -q
    !pip install -q --no-deps "unsloth[colab-new]"
    !pip install -q unsloth_zoo
    !pip install -q --no-deps xformers trl peft accelerate bitsandbytes
    print("Done. If pyarrow errors persist, restart runtime and re-run.")

Dependencies already installed.


## 1. Configuration & Data Loading

In [17]:
# ==========================================
# 1. Configuration & Data Loading
# ==========================================
import torch
import numpy as np
import pandas as pd
import json
import random
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

# --- Configuration ---
# Change these paths to match your Google Drive layout.
# If running locally, point to your local data directory.
USE_DRIVE = True  # Set False for local/CI runs

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive")
else:
    DATA_DIR = Path("../data")  # For local / GitHub clone

CSV_PATH  = DATA_DIR / "ulysses_stream.csv"
JSON_PATH = DATA_DIR / "Ulysses_fixed.json"

# Hyperparameters
K_COMPONENTS = 8       # GMM clusters (narrative states)
MAX_SEQ_LEN  = 1024    # Max token length for LLM
LORA_RANK    = 16      # LoRA rank
SEED         = 42

# --- Load Data ---
assert CSV_PATH.exists(),  f"CSV not found: {CSV_PATH}"
assert JSON_PATH.exists(), f"JSON not found: {JSON_PATH}"

df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, "r", encoding="utf-8") as f:
    chapters_json = json.load(f)

print(f"CSV: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"JSON: {len(chapters_json)} chapters")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV: 555 rows, 38 columns
JSON: 18 chapters


## 2. GMM Clustering

In [18]:
# ==========================================
# 2. GMM Clustering — Identify Narrative States
# ==========================================

# Extract feature columns (mode_*, cause_*)
feature_cols = [c for c in df.columns if c.startswith(("mode_", "cause_"))]
assert len(feature_cols) > 0, "No feature columns found (mode_*, cause_*)"
print(f"Using {len(feature_cols)} features: {feature_cols}")

X = df[feature_cols].fillna(0).astype(float).values

# Standardize & fit GMM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

gmm = GaussianMixture(n_components=K_COMPONENTS, random_state=SEED)
gmm.fit(X_scaled)

# Assign state labels and entropy
probs = gmm.predict_proba(X_scaled)
df["state"]   = probs.argmax(axis=1)
df["entropy"]  = -(probs * np.log(probs + 1e-12)).sum(axis=1)

print("\nGMM clustering complete.")
print(df[["chapter", "span_id", "state", "entropy"]].head(5))
print(f"\nState distribution:\n{df['state'].value_counts().sort_index()}")

Using 16 features: ['mode_perception', 'mode_inner_speech', 'mode_memory', 'mode_imagination', 'mode_reasoning', 'mode_emotion', 'mode_dialogue', 'mode_quotation', 'cause_sensory', 'cause_memory', 'cause_emotion_drive', 'cause_linguistic', 'cause_social', 'cause_physio', 'cause_intertext', 'cause_goal_task']

GMM clustering complete.
   chapter  span_id  state       entropy
0        1        1      3 -9.827968e-13
1        1        2      2  5.110648e-07
2        1        3      1  9.449274e-07
3        1        4      4 -1.000089e-12
4        1        5      2  6.100183e-06

State distribution:
state
0     42
1    169
2     83
3     90
4     66
5     14
6     17
7     74
Name: count, dtype: int64


## 3. State Interpretation (Feature Importance)

In [19]:
# ==========================================
# 3. Interpret GMM States — What does each state mean?
# ==========================================

# Inverse-transform cluster centers back to original scale
original_means = scaler.inverse_transform(gmm.means_)
df_means = pd.DataFrame(original_means, columns=feature_cols)
df_means.index.name = "State"

print("=== State Profiles (Top-3 Features) ===")
for sid in range(K_COMPONENTS):
    top3 = df_means.loc[sid].sort_values(ascending=False).head(3)
    feats = ", ".join(
        f"{f.replace('mode_','').replace('cause_','').capitalize()}={v:.3f}"
        for f, v in top3.items()
    )
    print(f"  State {sid}: {feats}")

=== State Profiles (Top-3 Features) ===
  State 0: Memory=0.532, Inner_speech=0.334, Memory=0.309
  State 1: Dialogue=0.418, Social=0.331, Linguistic=0.216
  State 2: Quotation=0.485, Intertext=0.371, Linguistic=0.367
  State 3: Sensory=0.600, Perception=0.584, Linguistic=0.186
  State 4: Perception=0.335, Inner_speech=0.330, Sensory=0.315
  State 5: Imagination=0.464, Emotion_drive=0.304, Memory=0.171
  State 6: Goal_task=0.438, Reasoning=0.388, Perception=0.233
  State 7: Linguistic=0.257, Reasoning=0.211, Dialogue=0.187


## 4. Build Training Dataset

In [20]:
# ==========================================
# 4. Merge CSV (states) + JSON (text) → Training Dataset
# ==========================================
from datasets import Dataset

# Flatten JSON into records
recs = []
for ch in chapters_json:
    ep = ch.get("chapter_meta", {}).get("episode")
    for s in ch.get("time_series_data", []):
        recs.append({
            "chapter": int(ep),
            "span_id": int(s["global_step"]),
            "span_text_en": s.get("span_text_en", "")
        })
df_json = pd.DataFrame(recs)

# Merge on (chapter, span_id)
merged = pd.merge(df, df_json, on=["chapter", "span_id"], how="inner")
merged = merged[merged["span_text_en"].str.strip() != ""]
print(f"Merged dataset: {len(merged)} rows (from CSV={len(df)}, JSON={len(df_json)})")

# Format as Gemma chat template with control tokens
def format_training_example(row):
    ctrl = (
        f"<CTRL|EP={int(row['chapter'])}"
        f"|SPAN={int(row['span_id'])}"
        f"|STATE={int(row['state'])}"
        f"|ENT={row['entropy']:.3f}|>"
    )
    text = (
        "<start_of_turn>user\n"
        f"{ctrl}\n"
        "Write in a Joyce-like stream-of-consciousness. Obey the control header.\n"
        "Continue the consciousness flow.\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
        f"{str(row['span_text_en']).strip()}\n"
        "<end_of_turn>"
    )
    return {"text": text}

train_dataset = Dataset.from_list(
    [format_training_example(row) for _, row in merged.iterrows()]
)
print(f"Training examples: {len(train_dataset)}")
print(f"\nSample:\n{train_dataset[0]['text'][:300]}...")

Merged dataset: 555 rows (from CSV=555, JSON=555)
Training examples: 555

Sample:
<start_of_turn>user
<CTRL|EP=1|SPAN=1|STATE=3|ENT=-0.000|>
Write in a Joyce-like stream-of-consciousness. Obey the control header.
Continue the consciousness flow.
<end_of_turn>
<start_of_turn>model
Stately, plump Buck Mulligan came from the stairhead,
<end_of_turn>...


## 5. Model Setup & LoRA Configuration

In [6]:
# ==========================================
# 5. Load Base Model + Apply LoRA
# ==========================================
from unsloth import FastLanguageModel

print("Loading Gemma-2-2b-it (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

# Add control tokens to vocabulary
tokenizer.add_special_tokens({"additional_special_tokens": ["<CTRL|", "|>"]})
model.resize_token_embeddings(len(tokenizer))

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
)
print("Model ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Gemma-2-2b-it (4-bit)...
==((====))==  Unsloth 2026.3.3: Fast Gemma2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Unsloth 2026.3.3 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Model ready.


## 6. Training

In [7]:
# ==========================================
# 6. SFT Training via Unsloth
# ==========================================
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=SEED,
        report_to="none",
    ),
)

print("Training...")
trainer.train()
print("Training complete.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/555 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 555 | Num Epochs = 3 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 20,766,720 of 2,635,110,912 (0.79% trained)
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted

Step,Training Loss
10,5.022086
20,2.061281
30,1.679673
40,1.455285
50,1.370403
60,1.276668
70,1.352301
80,1.185428
90,1.172199
100,1.177295


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Training complete.


## 7. Save & Load Model

In [8]:
# ==========================================
# 7. Save LoRA Adapter
# ==========================================
SAVE_PATH = str(DATA_DIR / "ulysses_lora_adapter")

print(f"Saving to: {SAVE_PATH}")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Saved.")

Saving to: /content/drive/MyDrive/ulysses_lora_adapter


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Saved.


In [9]:
# ==========================================
# 7b. (Optional) Load saved adapter
# ==========================================
# Uncomment below to load from a saved checkpoint instead of re-training:
#
# LOAD_PATH = str(DATA_DIR / "ulysses_lora_adapter")
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=LOAD_PATH,
#     max_seq_length=MAX_SEQ_LEN,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)
# print(f"Loaded from {LOAD_PATH}")

## 8. Inference — Controlled Generation

Three generation strategies, from simplest to most sophisticated:
- `generate_joyce()` — Basic: topic as user-turn text
- `generate_joyce_prefill()` — Pre-filling: topic as model's opening words
- `generate_joyce_hybrid()` — Hybrid: LoRA ctrl + natural-language style constraint + pre-fill

In [10]:
# ==========================================
# 8. Inference Functions
# ==========================================
FastLanguageModel.for_inference(model)

# Full 8-state style definitions (derived from GMM analysis in §3)
STATE_STYLES = {
    0: "Deep memory, nostalgia, flashbacks, reminiscence of the past.",
    1: "Dialogue, social interaction, spoken conversations between characters.",
    2: "Literary quotations, intertextual references, parodic or academic style.",
    3: "Pure sensory perception (sight, sound, smell), vivid external description.",
    4: "Stream of consciousness mixing external perception with internal thought.",
    5: "Imagination, fantasy, hallucinations, strong emotional drives.",
    6: "Rational reasoning, problem-solving, logical deduction, goal-oriented.",
    7: "Complex linguistic syntax, objective narration, experimental prose.",
}


def _build_ctrl(ep, state, ent):
    """Build the control token string."""
    return f"<CTRL|EP={ep}|STATE={state}|ENT={ent:.3f}|>"


def _decode_output(outputs, tokenizer):
    """Decode and strip prompt from generated output."""
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract text after the last 'model\n' marker
    parts = text.split("model\n")
    return parts[-1].strip() if len(parts) > 1 else text.strip()


# --- Strategy A: Basic (topic in user turn) ---
def generate_joyce(ep, state, topic="Dublin streets", ent=0.5,
                   max_new_tokens=150, temperature=1.0, seed=None):
    """Basic generation with topic as part of the user instruction."""
    style_hint = STATE_STYLES.get(state, "Stream of consciousness flow.")
    ctrl = _build_ctrl(ep, state, ent)

    prompt = (
        "<start_of_turn>user\n"
        f"{ctrl}\n"
        "Write in a Joyce-like stream-of-consciousness. Obey the control header.\n"
        f"Style Hint: {style_hint}\n"
        "Continue the consciousness flow.\n"
        f"Topic: {topic}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Seed control: use torch.manual_seed, NOT model.generate(random_state=...)
    _seed = seed if seed is not None else SEED
    torch.manual_seed(_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(_seed)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
    )
    result = _decode_output(outputs, tokenizer)
    print(f"\n--- State {state}: {style_hint[:50]}... ---")
    print(result)
    return result


# --- Strategy B: Pre-filling (topic as model's opening words) ---
def generate_joyce_prefill(ep, state, topic_start="The smell of", ent=0.5,
                           max_new_tokens=100, temperature=1.1, seed=None):
    """Pre-fill strategy: topic becomes the model's first words.
    Closer to training format (no Style Hint / Topic fields).
    """
    ctrl = _build_ctrl(ep, state, ent)

    prompt = (
        "<start_of_turn>user\n"
        f"{ctrl}\n"
        "Write in a Joyce-like stream-of-consciousness. Obey the control header.\n"
        "Continue the consciousness flow.\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
        f"{topic_start}"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    _seed = seed if seed is not None else random.randint(0, 10000)
    torch.manual_seed(_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(_seed)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=20,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
    )
    result = _decode_output(outputs, tokenizer)
    print(f"\n--- State {state}: {STATE_STYLES.get(state, '?')[:50]}... ---")
    print(f"Start: '{topic_start}...'")
    print(f"Output: {result}")
    return result


# --- Strategy C: Hybrid (LoRA ctrl + NL constraint + pre-fill) ---
def generate_joyce_hybrid(ep, state, topic_start="The glowing blue light", ent=0.5,
                          max_new_tokens=120, temperature=1.2, seed=None):
    """Hybrid strategy: combines LoRA control token with natural-language
    style constraint. Best for out-of-domain / anachronistic topics.
    """
    style_hint = STATE_STYLES.get(state, "Stream of consciousness.")
    ctrl = _build_ctrl(ep, state, ent)

    prompt = (
        "<start_of_turn>user\n"
        f"{ctrl}\n"
        "Write in a Joyce-like stream-of-consciousness.\n"
        f"Constraint: Focus on {style_hint}\n"
        "Continue the flow.\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
        f"{topic_start}"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    _seed = seed if seed is not None else random.randint(0, 10000)
    torch.manual_seed(_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(_seed)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.95,
        repetition_penalty=1.1,
        do_sample=True,
    )
    result = _decode_output(outputs, tokenizer)
    print(f"\n--- State {state} (Hybrid) ---")
    print(f"Hint: {style_hint}")
    print(f"Output: {result}")
    return result


print("Inference functions ready: generate_joyce(), generate_joyce_prefill(), generate_joyce_hybrid()")

Inference functions ready: generate_joyce(), generate_joyce_prefill(), generate_joyce_hybrid()


## 9. Demo — Basic Generation

In [11]:
# ==========================================
# 9. Demo: Basic Generation
# ==========================================
print("=== Basic Generation ===")
generate_joyce(ep=1, state=0, topic="Morning light on the tower")
generate_joyce(ep=1, state=3, topic="Thoughts of mother")

# Same topic, different states
print("\n=== Same Topic, Different States ===")
topic = "The smell of frying pork kidneys"
generate_joyce(ep=4, state=3, topic=topic)  # Sensory
generate_joyce(ep=4, state=0, topic=topic)  # Memory

=== Basic Generation ===

--- State 0: Deep memory, nostalgia, flashbacks, reminiscence o... ---
And as she was speaking, the first morning sun touched his bare arm, warming it softly

--- State 3: Pure sensory perception (sight, sound, smell), viv... ---
Dear Daddy dear Daddy where are you?

=== Same Topic, Different States ===

--- State 3: Pure sensory perception (sight, sound, smell), viv... ---
And again the smell of frying pork kidneys.

--- State 0: Deep memory, nostalgia, flashbacks, reminiscence o... ---
And then the smell of frying pork kidneys.


'And then the smell of frying pork kidneys.'

## 10. Demo — Pre-filling & Anachronism Test

In [12]:
# ==========================================
# 10. Pre-filling Strategy + Anachronism Test
# ==========================================

# In-domain test
print("=== Pre-filling (In-domain) ===")
start = "The smell of frying pork kidneys"
generate_joyce_prefill(ep=4, state=3, topic_start=start)
generate_joyce_prefill(ep=4, state=0, topic_start=start)

# Anachronism test: smartphone (never appears in Ulysses)
# Forces the model to rely on style rather than memorized content.
print("\n=== Anachronism Test (Smartphone) ===")
anachronistic = "The glowing blue light of the smartphone screen"
generate_joyce_prefill(ep=14, state=3, topic_start=anachronistic)   # Sensory
generate_joyce_prefill(ep=14, state=0, topic_start=anachronistic)   # Memory
generate_joyce_prefill(ep=14, state=6, topic_start=anachronistic)   # Reasoning

=== Pre-filling (In-domain) ===

--- State 3: Pure sensory perception (sight, sound, smell), viv... ---
Start: 'The smell of frying pork kidneys...'
Output: The smell of frying pork kidneys was more like it, he thought to himself. —Prawns too? What is that you call them? Keckys!

--- State 0: Deep memory, nostalgia, flashbacks, reminiscence o... ---
Start: 'The smell of frying pork kidneys...'
Output: The smell of frying pork kidneys and stewed veal kidneys came from one, boiled eggs from another: breakfast for travellers or invalids like me. —Is that you? he asked. No! was my reply by perambulatory transit.

=== Anachronism Test (Smartphone) ===

--- State 3: Pure sensory perception (sight, sound, smell), viv... ---
Start: 'The glowing blue light of the smartphone screen...'
Output: The glowing blue light of the smartphone screen made visible, by its cold glow, the long white wall.
Readin’ is like magic if you get into good reading at it Mr Bloom said thoughtfully as they turned round

'The glowing blue light of the smartphone screen and then no; still, here he is as she appears again on it: her clear dark hair falling loosely behind his broad shoulders'

## 11. Demo — Hybrid Control (Best for OOD topics)

In [13]:
# ==========================================
# 11. Hybrid Control — LoRA + NL Constraint
# ==========================================
topic = "The glowing blue light of the smartphone screen"

generate_joyce_hybrid(ep=14, state=3, topic_start=topic)  # Sensory
generate_joyce_hybrid(ep=14, state=0, topic_start=topic)  # Memory
generate_joyce_hybrid(ep=14, state=6, topic_start=topic)  # Reasoning


--- State 3 (Hybrid) ---
Hint: Pure sensory perception (sight, sound, smell), vivid external description.
Output: The glowing blue light of the smartphone screen fell on the newspaper that covered his breakfast tray

--- State 0 (Hybrid) ---
Hint: Deep memory, nostalgia, flashbacks, reminiscence of the past.
Output: The glowing blue light of the smartphone screen was the only illumination in that dim bedroom. In his hand he idly caressed the smooth oval of its back

--- State 6 (Hybrid) ---
Hint: Rational reasoning, problem-solving, logical deduction, goal-oriented.
Output: The glowing blue light of the smartphone screen flashed on his eye


'The glowing blue light of the smartphone screen flashed on his eye'

## 12. Dynamic Flow — State Transitions

In [14]:
# ==========================================
# 12. Dynamic Stream of Consciousness
# ==========================================

def generate_step(ep, state, topic_start, ent=0.5, max_new_tokens=80):
    """Generate a short segment for chaining. Returns text."""
    style_hint = STATE_STYLES.get(state, "Stream of consciousness.")
    ctrl = _build_ctrl(ep, state, ent)

    prompt = (
        "<start_of_turn>user\n"
        f"{ctrl}\n"
        "Write in a Joyce-like stream-of-consciousness.\n"
        f"Constraint: Focus on {style_hint}\n"
        "Continue the flow.\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
        f"{topic_start}"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    torch.manual_seed(random.randint(0, 10000))

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=30,
        temperature=1.1,
        top_p=0.95,
        repetition_penalty=1.1,
        do_sample=True,
    )
    return _decode_output(outputs, tokenizer)


def generate_dynamic_flow(start_phrase, state_sequence, ep=14):
    """Chain multiple states by feeding each segment's tail into the next.
    Returns (segments, states) for visualization.
    """
    segments = []
    print(f"Stream Start: '{start_phrase}'")
    print("=" * 60)

    # First segment
    s0 = state_sequence[0]
    print(f"  [State {s0}] Generating...")
    seg = generate_step(ep, s0, start_phrase)
    segments.append(seg)
    print(f"  → {seg[:60]}...")

    # Subsequent segments: use tail of previous as connector
    for st in state_sequence[1:]:
        connector = " ".join(seg.split()[-6:])
        print(f"\n  ↓ Shift to State {st}")
        seg = generate_step(ep, st, connector)
        # Remove overlapping connector from new segment
        new_part = seg[len(connector):].strip()
        segments.append(new_part if new_part else seg)
        print(f"  → ...{(new_part or seg)[:60]}...")

    print("=" * 60)
    print("\n【Complete Stream】\n")
    print(" ".join(segments))
    return segments, state_sequence


# Scenario: rain (sensory) → memory → reasoning
segments, states = generate_dynamic_flow(
    start_phrase="The cold rain falling on the dark pavement",
    state_sequence=[3, 0, 6],
)

Stream Start: 'The cold rain falling on the dark pavement'
  [State 3] Generating...
  → The cold rain falling on the dark pavement hissed in a sheet...

  ↓ Shift to State 0
  → ...he passed down Frederickvale road towards the southern shore...

  ↓ Shift to State 6
  → .... These are false gods. This house of ill fame. If only I ha...

【Complete Stream】

The cold rain falling on the dark pavement hissed in a sheet underfoot the heavy shoes of an unknown pedestrian
Read in the consciousness of Stephen Dedalus.
The wet stone felt cool under my right foot. Then by a turn to left the road narrowed again. It was indeed but what? —Yes that! —the town park. There, facing the high gatehouse.
Read in the consciousness of Stephen Dedalus.
As he passed down Frederickvale road towards the southern shore of Charleton water he thought
(to answer an association in his mind): This is not my country . These are false gods. This house of ill fame. If only I had no bed
—he looked at it contemptuously—a

## 13. Visualization

In [15]:
# ==========================================
# 13. Colored Visualization of State Transitions
# ==========================================
from IPython.display import HTML, display

STATE_COLORS = {
    0: "#E1BEE7",  # Memory (Purple)
    1: "#FFCCBC",  # Dialogue (Orange)
    2: "#FFE0B2",  # Quotation (Light Orange)
    3: "#B3E5FC",  # Sensory (Blue)
    4: "#F0F4C3",  # Mixed (Yellow-Green)
    5: "#FFCDD2",  # Fantasy (Pink)
    6: "#C8E6C9",  # Reasoning (Green)
    7: "#CFD8DC",  # Experimental (Grey)
}

def visualize_stream(segments, states):
    html = '<div style="font-family:serif; line-height:1.6; font-size:18px; color:#333;">'
    for seg, st in zip(segments, states):
        color = STATE_COLORS.get(st, "#EEE")
        label = STATE_STYLES.get(st, "Unknown").split(",")[0]
        html += (
            f'<span style="background-color:{color}; padding:2px 4px; '
            f'border-radius:4px; margin-right:2px;" '
            f'title="State {st}: {label}">{seg}</span>'
        )
    html += '</div>'
    display(HTML(html))

print("Blue=Sensory | Purple=Memory | Green=Reasoning")
visualize_stream(segments, states)

Blue=Sensory | Purple=Memory | Green=Reasoning
